# data_prep.py converted
Utilities to convert SMILES to PyG Data objects and save processed dataset.
Original script converted to a single code cell for interactive use.

In [3]:
"""
Data preparation utilities for EGFR GNN project.
Converts SMILES -> PyG Data objects and saves processed dataset.

This version produces:
- Node features: 6 floats per atom
- Edge features: 4 floats per bond (one-hot bond type)
- Global features: 2 floats per graph (num_atoms, num_bonds)
"""
from pathlib import Path
import logging
import warnings
from typing import Optional, List, Tuple

import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
from rdkit import Chem

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

warnings.filterwarnings('ignore')


def get_atomic_features(atom: Chem.Atom) -> np.ndarray:
    """Return 6 node features as floats:
    [atomic_num, degree, formal_charge, hybridization_ordinal, is_aromatic, num_explicit_hs]
    """
    atomic_num = float(atom.GetAtomicNum())
    degree = float(atom.GetDegree())
    formal_charge = float(atom.GetFormalCharge())
    hybridization_ordinal = float(atom.GetHybridization().real)
    is_aromatic = float(atom.GetIsAromatic())
    num_explicit_hs = float(atom.GetNumExplicitHs())

    return np.array([atomic_num, degree, formal_charge, hybridization_ordinal, is_aromatic, num_explicit_hs])


def get_bond_features(bond: Chem.Bond) -> np.ndarray:
    """Return 4 bond features as one-hot: [SINGLE, DOUBLE, TRIPLE, AROMATIC]"""
    bond_types = [Chem.BondType.SINGLE, Chem.BondType.DOUBLE, Chem.BondType.TRIPLE, Chem.BondType.AROMATIC]
    bond_type = bond.GetBondType()
    one_hot = np.array([1.0 if bond_type == bt else 0.0 for bt in bond_types])
    return one_hot


def smiles_to_pyg_data(smiles: str, label: float = 0.0, mol_id: str = None) -> Optional[Data]:
    """Convert a SMILES string to a PyG Data object."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        x = np.array([get_atomic_features(atom) for atom in mol.GetAtoms()], dtype=np.float32)
        x = torch.from_numpy(x)

        edges = mol.GetBonds()
        if len(edges) == 0:
            edge_index = torch.zeros((2, 0), dtype=torch.long)
            edge_attr = torch.zeros((0, 4), dtype=torch.float32)
        else:
            edge_list = [(bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()) for bond in edges]
            edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
            edge_attr = np.array([get_bond_features(bond) for bond in edges], dtype=np.float32)
            edge_attr = torch.from_numpy(edge_attr)

        num_atoms = float(mol.GetNumAtoms())
        num_bonds = float(mol.GetNumBonds())
        global_features = torch.tensor([num_atoms, num_bonds], dtype=torch.float32)

        data = Data(
            x=x,
            edge_index=edge_index,
            edge_attr=edge_attr,
            y=torch.tensor(label, dtype=torch.float32),
            global_features=global_features,
            smiles=smiles,
            mol_id=mol_id
        )

        return data

    except Exception as e:
        logger.warning(f"Failed to convert SMILES {smiles}: {e}")
        return None


def load_and_clean(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    if 'Smiles' not in df.columns or 'pChEMBL_Value' not in df.columns:
        raise ValueError("CSV must contain 'Smiles' and 'pChEMBL_Value' columns")
    df = df.dropna(subset=['Smiles', 'pChEMBL_Value'])
    df = df.drop_duplicates(subset=['Smiles'])
    df['pChEMBL_Value'] = pd.to_numeric(df['pChEMBL_Value'], errors='coerce')
    df = df.dropna(subset=['pChEMBL_Value']).reset_index(drop=True)
    return df


def process(csv_path: str, out_path: str, max_mols: Optional[int] = None) -> Tuple[List[Data], dict]:
    df = load_and_clean(csv_path)
    if max_mols:
        df = df.head(max_mols)
    graphs: List[Data] = []
    failed = 0
    for i, row in df.iterrows():
        data = smiles_to_pyg_data(row['Smiles'], float(row['pChEMBL_Value']), mol_id=str(i))
        if data is None:
            failed += 1
        else:
            graphs.append(data)
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(graphs, out_path)
    stats = {'total': len(df), 'success': len(graphs), 'failed': failed, 'out': out_path}
    return graphs, stats


# In notebook environment, skip argparse execution
print("data_prep utility functions defined successfully")

data_prep utility functions defined successfully
